# Qwen-Image-Edit-2509 OCR Experiment

This notebook demonstrates an experiment where we use Qwen-Image-Edit-2509 to stylize text in an image, then use an OCR model to detect and extract the text from the modified image, and finally compare the detected text with the original text.

In [ ]:
from difflib import SequenceMatcher
from pathlib import Path

try:
    import polars as pl
except ImportError as exc:
    raise RuntimeError("Polars is required. Run `uv sync` (or `pip install polars`) before executing this notebook.") from exc
import torch
from PIL import Image
try:
    from diffusers import QwenImageEditPlusPipeline
except ImportError as exc:
    raise RuntimeError("diffusers>=0.27.2 is required. Run `uv sync` (or `pip install diffusers>=0.27.2`).") from exc
from IPython.display import display

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
DATA_DIR = Path("data")


## Load Model and Input Image

First, we'll load the Qwen-Image-Edit-2509 pipeline and move it to the GPU. We'll also load the input image needed for the experiment.

In [ ]:
MODEL_ID = "Qwen/Qwen-Image-Edit-2509"
pipeline = QwenImageEditPlusPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16
)
pipeline.to("cuda")
pipeline.set_progress_bar_config(disable=None)
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"Pipeline {MODEL_ID} loaded on {device_name}")

def load_image(name: str) -> Image.Image:
    search_paths = [DATA_DIR / name, Path(name)]
    for candidate in search_paths:
        if candidate.exists():
            return Image.open(candidate).convert("RGB")
    raise FileNotFoundError(f"Unable to find {name}; place it under data/ or the repo root.")

input_image = load_image("ocr_input.png")
print("Input image loaded")


## Stylize Text in Image

We'll use the Qwen-Image-Edit-2509 model to transform all English text in the image to be stylized and fully capitalized.

In [ ]:
prompt = "Make every English word appear as fully capitalized, stylized lettering."

inputs = {
    "image": input_image,
    "prompt": prompt,
    "generator": torch.manual_seed(0),
    "true_cfg_scale": 4.0,
    "negative_prompt": " ",
    "num_inference_steps": 40,
    "guidance_scale": 1.0,
    "num_images_per_prompt": 1,
}

with torch.inference_mode():
    output = pipeline(**inputs)
stylized_image = output.images[0]
stylized_image_path = RESULTS_DIR / "stylized_image.png"
stylized_image.save(stylized_image_path)
print(f"Stylized image saved at {stylized_image_path.resolve()}")


## Extract Text with OCR

Now we'll use a state-of-the-art OCR model to extract text from the stylized image. For this experiment, we'll use EasyOCR which is a popular choice for text detection.

In [ ]:
try:
    import easyocr
except ImportError as exc:
    raise RuntimeError("Install easyocr via `uv pip install easyocr` before running this cell.") from exc

reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
print("OCR reader initialized")
ocr_results = reader.readtext(str(stylized_image_path))
ocr_rows = [
    {"index": idx + 1, "text": text, "confidence": confidence}
    for idx, (_, text, confidence) in enumerate(ocr_results)
]
ocr_df = pl.DataFrame(ocr_rows) if ocr_rows else pl.DataFrame({"index": [], "text": [], "confidence": []})
display(ocr_df)
detected_text = " ".join(row['text'] for row in ocr_rows) if ocr_rows else ""
detected_text_path = RESULTS_DIR / "ocr_detected_text.txt"
detected_text_path.write_text(detected_text)
print(f"Detected text saved to {detected_text_path.resolve()}")


## Compare Text and Calculate Match Percentage

Finally, we'll compare the detected text with the original text stored in control.txt and calculate the percentage match. We need to ensure control.txt exists in the working directory with the original text content.

In [ ]:
def load_reference_text() -> str:
    candidates = [DATA_DIR / "control.txt", Path("control.txt")]
    for candidate in candidates:
        if candidate.exists():
            return candidate.read_text().strip()
    raise FileNotFoundError("control.txt not found; add it under data/ or the repo root.")

try:
    original_text = load_reference_text()
except FileNotFoundError as exc:
    print(exc)
    original_text = ""
    print("Proceeding with empty reference text.")

match_ratio = SequenceMatcher(None, original_text.lower(), detected_text.lower()).ratio()
match_percentage = match_ratio * 100
comparison_df = pl.DataFrame(
    {
        "metric": ["match_percentage", "original_length", "detected_length"],
        "value": [match_percentage, len(original_text), len(detected_text)],
    }
)
display(comparison_df)
comparison_path = RESULTS_DIR / "ocr_match_summary.parquet"
comparison_df.write_parquet(comparison_path)
print(f"Match summary saved to {comparison_path.resolve()}")
